In [1]:
import qbiome
from qbiome.data_formatter import DataFormatter
from qbiome.quantizer import Quantizer
from qbiome.qnet_orchestrator import QnetOrchestrator
from qbiome.forecaster import Forecaster
from qbiome.hypothesis import Hypothesis

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use("fivethirtyeight")
from qbiome.qutil import qsmooth


In [2]:
# File path to your merged CSV file
file_path = "example_data/training_data_weeks_60_to_90_sewage_timeseries.csv"  # update with actual path if needed

# Read the CSV into a DataFrame
merged_df = pd.read_csv(file_path,keep_default_na=False)

# Preview the first few rows
print(merged_df.head())

   sample_id     Site          taxa_id  rel_abundance      COLLECTION_DATE  \
0  1014220_1  Bologna       Acidovorax       0.151028  2020-04-07 00:00:00   
1  1014220_1  Bologna    Acinetobacter       0.017902  2020-04-07 00:00:00   
2  1014220_1  Bologna        Aeromonas       0.004478  2020-04-07 00:00:00   
3  1014220_1  Bologna    Aliarcobacter       0.006025  2020-04-07 00:00:00   
4  1014220_1  Bologna  Aquabacterium_A       0.067315  2020-04-07 00:00:00   

                               COMPLETE_NAME COUNTRY     CITY  LATITUDE  \
0  DTU_2022_1014220_1_MG_IT_BO_200407_GSL_60   Italy  Bologna   44.4949   
1  DTU_2022_1014220_1_MG_IT_BO_200407_GSL_60   Italy  Bologna   44.4949   
2  DTU_2022_1014220_1_MG_IT_BO_200407_GSL_60   Italy  Bologna   44.4949   
3  DTU_2022_1014220_1_MG_IT_BO_200407_GSL_60   Italy  Bologna   44.4949   
4  DTU_2022_1014220_1_MG_IT_BO_200407_GSL_60   Italy  Bologna   44.4949   

   LONGITUDE        PLANT  relative_week  
0    11.3426  Gruppo HERA            

In [3]:
merged_df = merged_df[merged_df['taxa_id'] != 'merged']

In [4]:
# Select the desired columns
data = merged_df[['sample_id', 'Site', 'taxa_id', 'rel_abundance', 'relative_week']].copy()

# Rename the columns
data.rename(columns={
    'Site': 'subject_id',
    'taxa_id': 'variable',
    'rel_abundance': 'value',
    'relative_week': 'week'
}, inplace=True)

# Preview the result
print(data.head())

   sample_id subject_id         variable     value  week
0  1014220_1    Bologna       Acidovorax  0.151028    61
1  1014220_1    Bologna    Acinetobacter  0.017902    61
2  1014220_1    Bologna        Aeromonas  0.004478    61
3  1014220_1    Bologna    Aliarcobacter  0.006025    61
4  1014220_1    Bologna  Aquabacterium_A  0.067315    61


In [5]:
# Standardize column names for Qbiome
data.rename(columns={
    "Site": "subject_id",
    "taxa_id": "variable",
    "rel_abundance": "value",
    "relative_week": "week"
}, inplace=True)

In [6]:
# STEP 1: Pad future weeks (91-120) with missing values
subjects = data['subject_id'].unique()
taxa = data['variable'].unique()
future_weeks = range(91, 121)

In [7]:
import uuid

# Pad future weeks (91–120)
pad = pd.MultiIndex.from_product(
    [subjects, range(91, 121), taxa],
    names=['subject_id', 'week', 'variable']
).to_frame(index=False)

# Add placeholder values for 'value' and 'sample_id'
pad['value'] = -1.0  # Placeholder/masked value
pad['sample_id'] = pad['subject_id'] + "_future_" + pad['week'].astype(str)

# OPTIONAL: If you want a completely random UUID instead:
# pad['sample_id'] = [str(uuid.uuid4()) for _ in range(len(pad))]

# Reorder columns to match the original data
pad = pad[['sample_id', 'subject_id', 'variable', 'value', 'week']]


In [8]:
combined_data = pd.concat([data, pad], ignore_index=True)

In [9]:
combined_data

,sample_id,subject_id,variable,value,week
0,1014220_1,Bologna,Acidovorax,0.151028,61
1,1014220_1,Bologna,Acinetobacter,0.017902,61
2,1014220_1,Bologna,Aeromonas,0.004478,61
3,1014220_1,Bologna,Aliarcobacter,0.006025,61
4,1014220_1,Bologna,Aquabacterium_A,0.067315,61
...,...,...,...,...,...
14295,Rotterdam_future_120,Rotterdam,Thermomonas,-1.000000,120
14296,Rotterdam_future_120,Rotterdam,Thiothrix,-1.000000,120
14297,Rotterdam_future_120,Rotterdam,Tolumonas,-1.000000,120
14298,Rotterdam_future_120,Rotterdam,Trichococcus,-1.000000,120


In [10]:
# STEP 2: Quantize
quantizer = Quantizer(num_levels=10)
quantized_df = quantizer.quantize_df(combined_data)

In [11]:

qnet_orchestrator = QnetOrchestrator(quantizer)

In [12]:
features, label_matrix = quantizer.get_qnet_inputs(quantized_df)

In [13]:
display(features)
display(label_matrix)

Index(['Acidovorax_60', 'Acidovorax_61', 'Acidovorax_62', 'Acidovorax_63',
       'Acidovorax_64', 'Acidovorax_65', 'Acidovorax_66', 'Acidovorax_67',
       'Acidovorax_68', 'Acidovorax_69',
       ...
       'UBA1413_111', 'UBA1413_112', 'UBA1413_113', 'UBA1413_114',
       'UBA1413_115', 'UBA1413_116', 'UBA1413_117', 'UBA1413_118',
       'UBA1413_119', 'UBA1413_120'],
      dtype='object', length=2640)

array([['', 'J', '', ..., 'E', 'E', 'E'],
       ['', '', '', ..., 'E', 'E', 'E'],
       ['', 'A', '', ..., 'E', 'E', 'E'],
       ...,
       ['J', 'J', '', ..., 'E', 'E', 'E'],
       ['G', '', 'J', ..., 'E', 'E', 'E'],
       ['', '', 'A', ..., 'E', 'E', 'E']], dtype='<U1')

In [15]:
base_tax = "Time_Series_Sewage_time_split_v2"

In [ ]:
%%time
qnet_orchestrator.train_qnet(
    features, label_matrix, alpha=0.3, min_samples_split=2, out_fname=None
)
qnet_orchestrator.save_qnet("example_qnet_"+base_tax+".pkl",GZIP=True)

In [ ]:
qnet_orchestrator = QnetOrchestrator(quantizer)
qnet_orchestrator.load_qnet("example_qnet_" + base_tax + ".pkl.gz", GZIP=True)

In [ ]:
# forecast from time SW
SW = 60
qnet_orchestrator = QnetOrchestrator(quantizer)
qnet_orchestrator.load_qnet("example_qnet_Genus.pkl.gz",GZIP=True)
forecaster = Forecaster(qnet_orchestrator)
# can specify an end week or default to the max end week in the data
forecasted_ = forecaster.forecast_data(label_matrix, start_week=SW).assign(
    source="forecasted"
)

In [ ]:
DF = forecasted_
SUBJECTS = DF.subject_id.value_counts().index.values
DF

DF_pop = DF.groupby(["variable", "week"]).mean(numeric_only=True).reset_index()
DF_data = data.groupby(["variable", "week"]).mean(numeric_only=True).reset_index()

In [ ]:
DF_pop = DF.groupby(["variable", "week"]).mean(numeric_only=True).reset_index()
DF_data = data.groupby(["variable", "week"]).mean(numeric_only=True).reset_index()

DF_pop_smooth = qsmooth(
    DF_pop,
    index="week",
    columns="variable",
    normalize=False,
    alpha=0.99,
    interpolate=False,
    lowess_fraction=0.3,
)

# DF_data_smooth
DF_data_smooth = qsmooth(
    DF_data,
    index="week",
    columns="variable",
    normalize=False,
    alpha=0.99,
    interpolate=False,
    lowess_fraction=0.3,
)

ax = DF_pop_smooth[biome].plot(label="forecast")
DF_data_smooth[biome].plot(ax=ax, label="observation")
ax.set_title(biome)
ax.legend()